In [1]:
import pandas as pd
import numpy as np
# import os # might be neede for file handling later.

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/'
print("Libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Libraries loaded successfully!
Pandas version: 3.0.3
Numpy version: 2.4.4


In [2]:
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
geo = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv',
                     parse_dates=['order_purchase_timestamp',
                                  'order_approved_at',
                                  'order_delivered_carrier_date',
                                  'order_delivered_customer_date',
                                  'order_estimated_delivery_date'])
products = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
categories = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print("All 9 tables loaded successfully!")

All 9 tables loaded successfully!


In [3]:
tables = {
    'customers': customers,
    'geolocation': geo,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'categories': categories
}

for name, df in tables.items():
    print(f"\n{'='*45}")
    print(f" {name.upper()}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Nulls: {df.isnull().sum().sum():,}")
    print(f"Columns: {list(df.columns)}")


 CUSTOMERS
Rows: 99,441
Columns: 5
Nulls: 0
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

 GEOLOCATION
Rows: 1,000,163
Columns: 5
Nulls: 0
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

 ORDER_ITEMS
Rows: 112,650
Columns: 7
Nulls: 0
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

 PAYMENTS
Rows: 103,886
Columns: 5
Nulls: 0
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

 REVIEWS
Rows: 99,224
Columns: 7
Nulls: 145,903
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

 ORDERS
Rows: 99,441
Columns: 8
Nulls: 4,908
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_

In [4]:
print("Null counts per column:\n")
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"\n{name.upper()}:")
        for col in nulls.index:
            count = int(nulls[col])
            pct = count / len(df) * 100
            print(f"  {col}: {count:,} ({pct:.1f}%)")
        print() 

Null counts per column:


REVIEWS:
  review_comment_title: 87,656 (88.3%)
  review_comment_message: 58,247 (58.7%)


ORDERS:
  order_approved_at: 160 (0.2%)
  order_delivered_carrier_date: 1,783 (1.8%)
  order_delivered_customer_date: 2,965 (3.0%)


PRODUCTS:
  product_category_name: 610 (1.9%)
  product_name_lenght: 610 (1.9%)
  product_description_lenght: 610 (1.9%)
  product_photos_qty: 610 (1.9%)
  product_weight_g: 2 (0.0%)
  product_length_cm: 2 (0.0%)
  product_height_cm: 2 (0.0%)
  product_width_cm: 2 (0.0%)



In [5]:
print("Date columns parsed successfully!\n")
print(orders[['order_purchase_timestamp', 
              'order_approved_at', 
              'order_delivered_carrier_date',
              'order_delivered_customer_date',
              'order_estimated_delivery_date']].dtypes)
print("\nDate range:")
print(f"  Earliest order: {orders['order_purchase_timestamp'].min()}")
print(f"  Latest order: {orders['order_purchase_timestamp'].max()}")

Date columns parsed successfully!

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Date range:
  Earliest order: 2016-09-04 21:15:19
  Latest order: 2018-10-17 17:30:18


## Phase 1 Summary

### What we did
- Set up project environment on Mac (Python 3.14.4, venv, all packages)
- Downloaded Olist dataset via Kaggle API (9 CSV files, 100k+ orders)
- Loaded all 9 tables into pandas and performed initial data audit
- Identified null values and their locations across all tables
- Parsed 5 date columns using parse_dates during read_csv

### What we found
- 5 out of 9 tables are completely clean (zero nulls)
- Reviews nulls (88.3%) are expected — customers rarely write comments
- Orders nulls (max 3.0%) belong to cancelled/undelivered orders
- Products have 610 incomplete listings needing attention
- Dataset spans September 2016 to October 2018 (2+ years)

### What's next (Phase 2)
- Rename typo columns (product_name_lenght → product_name_length)
- Fill missing product categories with 'Unknown'
- Handle orders nulls strategically (keep, don't drop)
- Remove geolocation duplicates
- Build master orders table by joining all key tables